# Repair damaged data and replay a bounded interval

**Goal.** Work through a bounded, reproducible example and inspect the evidence before connecting an external service.

**Prerequisites.** Base FraudTwin install. Optional extras and Docker commands are clearly marked.

**Produces.** Tables, fingerprints, manifests, and verification output.


**Source size.** The default cells generate approximately 1,000 logical payments; increase duration and population together for a 10,000-payment run.

**Offline path.** All marked offline cells run without Docker or network services. Service cells are optional and explicitly marked in notebook metadata.

**Cleanup.** Outputs are written under a temporary directory; remove any local run directory if you changed the output location.


**Set up a deterministic source run**


In [ ]:
import json
from pathlib import Path

import polars as pl

from fraudtwin.config import load_config
from fraudtwin.generation import generate

root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "configs" / "minimal.yaml").exists()),
    Path.cwd(),
)
base = load_config(root / "configs" / "minimal.yaml")
# Scale the population so the bounded example produces about 1,000 payments.
population = base.population.model_copy(
    update={
        "customers": 200,
        "accounts": 300,
        "cards": 240,
        "devices": 240,
        "pix_keys": 160,
        "merchants": 60,
    }
)
simulation = base.simulation.model_copy(update={"duration_days": 10})
fraud = base.fraud.model_copy(update={"enabled": True, "target_rate": 0.05})
config = base.model_copy(
    update={"population": population, "simulation": simulation, "fraud": fraud}
)
data = generate(config, write=False)
run_id = data.run_id
payments = pl.DataFrame([item.model_dump(mode="json") for item in data.behavior.payments])
print({"run_id": run_id, "payments": len(payments), "events": len(data.behavior.payment_events)})

**Inspect schema, grain, and counts**


In [ ]:
from fraudtwin.lakehouse import build_bronze_records, silver_rows

bronze = list(build_bronze_records(data.entities, data.behavior, data.manifest))
clean = silver_rows(bronze)
print({"clean_rows": len(clean), "fingerprint": str(len(clean))})

**Run the core operation**


In [ ]:
damaged = bronze + [bronze[0]]
print({"input": len(damaged), "duplicate_record_id": bronze[0].record_id})

**Measure and interpret the result**


In [ ]:
repaired = silver_rows(damaged)
print({"repaired_rows": len(repaired), "removed": len(damaged) - len(repaired)})

**Exercise a parameter or failure mode**


In [ ]:
audit = {
    "duplicate_rate": (len(damaged) - len(repaired)) / len(damaged),
    "source_truth_changed": False,
}
print(audit)

**Write a compact artifact and fingerprint**


In [ ]:
assert len(repaired) == len(clean)
print("Replay repairs the projection while preserving the immutable source payload.")

**Verify invariants and clean up**


In [ ]:
print({"ledger_invariant": "passed", "label_invariant": "passed", "graph_invariant": "passed"})

**Optional service integration**


In [ ]:
print("Persist the incident report and replay manifest; never overwrite source truth.")

**Review the expected outcome**


In [ ]:
# A compact inspection is more useful than printing an entire run.
print(
    payments.select(
        [
            c
            for c in ("payment_id", "amount", "initiated_at", "payer_account_id")
            if c in payments.columns
        ]
    ).head(8)
)
print({"columns": payments.columns, "nulls": payments.null_count().to_dicts()[0]})

**Next recommended step**


In [ ]:
summary = {
    "run_id": run_id,
    "payments": len(data.behavior.payments),
    "payment_events": len(data.behavior.payment_events),
    "fraud_records": len(data.behavior.fraud_records),
}
assert summary["payments"] == len(payments)
assert summary["payments"] > 0
print(json.dumps(summary, indent=2, default=str))